In [1]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/application_train_features.csv")

preprocessor = joblib.load("models/preprocessor.pkl")
lgbm_model = joblib.load("models/lightgbm_model.pkl")

X = df.drop(columns="TARGET")
y = df["TARGET"]

X_processed = preprocessor.transform(X)
y_pred = lgbm_model.predict(X_processed)
y_prob = lgbm_model.predict_proba(X_processed)[:, 1]

df['PREDICTED'] = y_pred
df['PREDICTED_PROB'] = y_prob

/Users/swarnimsingh/Library/Python/3.11/lib/python/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/swarnimsingh/Library/Python/3.11/lib/python/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_54665/3549617755.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['PREDICTED'] = y_pred
/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_54665/3549617755.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.in

In [2]:
df['AGE_GROUP'] = pd.cut(
    df['AGE_YEARS'],
    bins=[0, 30, 50, 100],
    labels=['Under 30', '30-50', '50+']
)

/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_54665/139242174.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE_GROUP'] = pd.cut(


In [4]:
def disparate_impact(df, group_col, group_a, group_b, pred_col='PREDICTED'):
    """
    Positive prediction here = predicted default (1) — i.e., flagged as risky.
    """
    rate_a = df[df[group_col] == group_a][pred_col].mean()
    rate_b = df[df[group_col] == group_b][pred_col].mean()
    di = rate_a / rate_b
    return rate_a, rate_b, di

rate_f, rate_m, di_gender = disparate_impact(df, 'CODE_GENDER', 'F', 'M')
print(f"Female flagged-as-risky rate: {rate_f:.4f}")
print(f"Male flagged-as-risky rate: {rate_m:.4f}")
print(f"Disparate Impact Ratio (F/M): {di_gender:.4f}")

Female flagged-as-risky rate: 0.2672
Male flagged-as-risky rate: 0.4349
Disparate Impact Ratio (F/M): 0.6145


In [5]:
def equal_opportunity_difference(df, group_col, group_a, group_b, 
                                    target_col='TARGET', pred_col='PREDICTED'):
    """
    True Positive Rate (recall) for each group, among actual defaulters only.
    """
    def tpr(group_df):
        actual_positives = group_df[group_df[target_col] == 1]
        if len(actual_positives) == 0:
            return np.nan
        return actual_positives[pred_col].mean()
    
    tpr_a = tpr(df[df[group_col] == group_a])
    tpr_b = tpr(df[df[group_col] == group_b])
    eod = tpr_a - tpr_b
    return tpr_a, tpr_b, eod

tpr_f, tpr_m, eod_gender = equal_opportunity_difference(df, 'CODE_GENDER', 'F', 'M')
print(f"Female recall (TPR): {tpr_f:.4f}")
print(f"Male recall (TPR): {tpr_m:.4f}")
print(f"Equal Opportunity Difference (F-M): {eod_gender:.4f}")

Female recall (TPR): 0.6566
Male recall (TPR): 0.7978
Equal Opportunity Difference (F-M): -0.1412


In [6]:
groups = df['AGE_GROUP'].unique()
print("=== Disparate Impact — Age Groups ===")
for g in groups:
    rate = df[df['AGE_GROUP'] == g]['PREDICTED'].mean()
    print(f"{g}: flagged-as-risky rate = {rate:.4f}")

print("\n=== Equal Opportunity — Age Groups ===")
for g in groups:
    subset = df[(df['AGE_GROUP'] == g) & (df['TARGET'] == 1)]
    tpr = subset['PREDICTED'].mean() if len(subset) > 0 else np.nan
    print(f"{g}: recall (TPR) = {tpr:.4f}")

=== Disparate Impact — Age Groups ===
Under 30: flagged-as-risky rate = 0.5336
30-50: flagged-as-risky rate = 0.3525
50+: flagged-as-risky rate = 0.1908

=== Equal Opportunity — Age Groups ===
Under 30: recall (TPR) = 0.8561
30-50: recall (TPR) = 0.7435
50+: recall (TPR) = 0.5352


### Fairness Audit Findings

**Disparate Impact:**
- **Gender:** **0.6145** — **below** the commonly used **0.8** fairness threshold, indicating that the model flags male and female applicants as high risk at different rates.
- **Age:** The model flagged **53.36%** of applicants **under 30**, **35.25%** of applicants aged **30–50**, and **19.08%** of applicants aged **50+** as high risk, indicating noticeable differences in prediction rates across age groups.

**Equal Opportunity Difference:**
- **Gender:** **-0.1412** — the model catches approximately **14.12% fewer** actual defaulters among **female** applicants than **male** applicants.
- **Age:** Recall (True Positive Rate) was **85.61%** for applicants **under 30**, **74.35%** for applicants aged **30–50**, and **53.52%** for applicants aged **50+**, showing that the model identifies actual defaulters more effectively in younger applicants than in older applicants.

**Discussion:**
[Your interpretation — e.g., is this disparity likely a reflection of real 
underlying risk differences in the data, or a sign of the model amplifying 
historical bias in who got approved/denied loans historically? What would 
you do about it — recalibrate thresholds per group, remove the feature 
entirely, or flag for human review?]

**Limitations:** This audit uses TARGET and PREDICTED labels only; it 
doesn't account for intersectional fairness (e.g. age × gender combined) 
or examine whether CODE_GENDER itself should be used as a model input at all.